# Project 20 — Local Medical Literature Finder
## Search Papers by Topic with Evidence Grading

**Stack:** Ollama · LlamaIndex · Metadata Filters · Jupyter

In [ ]:
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Setup

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

## Step 2 — Sample Medical Literature

In [ ]:
papers = [
    Document(text="""Title: Efficacy of Metformin in Type 2 Diabetes Management
Authors: Chen et al., 2024 | Journal: Lancet Diabetes
Study Type: Randomized Controlled Trial | Evidence Level: I
Sample Size: 2,400 patients | Duration: 24 months

Findings: Metformin reduced HbA1c by 1.2% compared to placebo (p<0.001).
Secondary outcomes showed 15% reduction in cardiovascular events.
Adverse effects: GI symptoms in 20% of patients (vs 8% placebo).
Conclusion: Metformin remains first-line therapy for T2DM.""",
        metadata={"topic": "diabetes", "year": 2024, "evidence_level": "I",
                 "study_type": "RCT", "journal": "Lancet"}),

    Document(text="""Title: Machine Learning for Early Cancer Detection
Authors: Park et al., 2024 | Journal: Nature Medicine
Study Type: Retrospective Cohort | Evidence Level: III
Sample Size: 50,000 imaging studies | Duration: 5 years

Findings: Deep learning model detected early-stage lung cancer with
94% sensitivity and 88% specificity, outperforming radiologists (87%/85%).
False positive rate was 12%, requiring further investigation.
Conclusion: AI-assisted screening shows promise but needs prospective validation.""",
        metadata={"topic": "oncology", "year": 2024, "evidence_level": "III",
                 "study_type": "retrospective", "journal": "Nature Medicine"}),

    Document(text="""Title: Cognitive Behavioral Therapy vs SSRIs for Anxiety
Authors: Williams et al., 2023 | Journal: JAMA Psychiatry
Study Type: Meta-Analysis | Evidence Level: I
Studies Included: 45 RCTs | Total Patients: 12,000

Findings: CBT and SSRIs showed equivalent efficacy at 12 weeks.
CBT showed lower relapse rate at 12 months (15% vs 30%, p<0.01).
Combined therapy showed 25% better outcomes than either alone.
Conclusion: CBT should be offered as first-line, alone or combined.""",
        metadata={"topic": "psychiatry", "year": 2023, "evidence_level": "I",
                 "study_type": "meta-analysis", "journal": "JAMA"}),
]
print(f"Loaded {len(papers)} medical papers")

## Step 3 — Build Literature Index

In [ ]:
index = VectorStoreIndex.from_documents(papers, show_progress=True)
query_engine = index.as_query_engine(similarity_top_k=3)
print("Literature index ready!")

## Step 4 — Evidence-Based Search

In [ ]:
queries = [
    "What is the first-line treatment for type 2 diabetes?",
    "How effective is AI for cancer screening?",
    "Compare CBT and medication for anxiety disorders",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Clinical Q: {q}")
    response = query_engine.query(q)
    print(f"\nEvidence Summary: {response}")
    print("\nSources:")
    for node in response.source_nodes:
        print(f"  [{node.metadata.get('evidence_level', '?')}] "
              f"{node.metadata.get('journal', '?')} "
              f"({node.metadata.get('study_type', '?')}, "
              f"{node.metadata.get('year', '?')})")

## Step 5 — Evidence Quality Grading

In [ ]:
from pydantic import BaseModel, Field

class EvidenceGrade(BaseModel):
    grade: str = Field(description="A (strong), B (moderate), C (weak), D (very weak)")
    reasoning: str
    level_of_evidence: str
    recommendation_strength: str
    caveats: list[str]

# Use ChatOllama directly for structured grading
from langchain_ollama import ChatOllama
grading_llm = ChatOllama(model="qwen3:8b", temperature=0.1)
grader = grading_llm.with_structured_output(EvidenceGrade)

for paper in papers:
    grade = grader.invoke(
        f"Grade the evidence quality of this study:\n\n{paper.text}"
    )
    title = paper.text.split("\n")[0].replace("Title: ", "")
    print(f"\n{title}")
    print(f"  Grade: {grade.grade}")
    print(f"  Evidence Level: {grade.level_of_evidence}")
    print(f"  Recommendation: {grade.recommendation_strength}")
    print(f"  Caveats: {grade.caveats}")

## What You Learned
- **Medical literature indexing** with evidence metadata
- **Evidence-based retrieval** with quality indicators
- **Structured evidence grading** using medical standards